In [1]:
pip install --upgrade --quiet langchain-core langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pydantic import BaseModel
from openai import AzureOpenAI
from langchain_openai import OpenAI

api_key = ""
api_endpoint = "" 

os.environ["OPENAI_API_KEY"] =''
os.environ["AZURE_OPENAI_API_KEY"] = api_key
os.environ["AZURE_OPENAI_ENDPOINT"] = api_endpoint

model_name= "gpt-4.1"
api_version = '2024-12-01-preview'

In [3]:
prompt_file = "prompt_4.1_optimized.txt"

dev_dataset_path = 'final_train_dataset.tsv'
predicted_results_path = f'pred_dev_gpt41_langchain_{model_name}_few_shot.tsv'
predicted_entities_path = f'pred_dev_gpt41_langchain_{model_name}_few_shot_entities.tsv'

In [5]:
from ast import literal_eval

prompt, examples, patient_text = '', '', ''

with open(prompt_file, 'r', encoding="utf-8") as file:
    prompt = file.read()

In [6]:
prompt

'```\nYou are a highly specialized AI, an expert in Biomedical Natural Language Processing (NLP) with a focus on extracting information related to substance abuse from Spanish clinical text. Your primary task is Named Entity Recognition (NER), specifically identifying and categorizing mentions of Tobacco, Cannabis, Alcohol, and illicit/abused prescription Drugs within the provided text. You must also determine if the mention is positive or negated based on the sentence context. Your output should be a JSON formatted list of dictionaries.\nHere\'s the detailed breakdown of your process and the required output format:\n---\n## Task: Named Entity Recognition (NER) for Substance Abuse in Spanish Clinical Text with Negation Detection\n**Goal:** To accurately identify, categorize, and determine the sentiment (positive or negated) of named entities related to substance abuse within a given Spanish clinical text.\n**Entity Categories (Substance Abuse Focus):**\n*   **Tobacco:** Mentions of tob

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

tagging_prompt = ChatPromptTemplate.from_template(prompt)

In [8]:
from langchain_openai import AzureChatOpenAI
from langchain.globals import set_verbose
from langchain_openai import ChatOpenAI

if os.environ["OPENAI_API_KEY"] == '': #use azure deployment
    llm = AzureChatOpenAI(
        azure_deployment=model_name,
        api_version=api_version,
        temperature=1,
        max_tokens=16000,
        top_p=1,
        timeout=None,
        max_retries=2,
    ).with_structured_output(method="json_mode")
else:    
    llm = ChatOpenAI(
        model=model_name,
        temperature=1,
        top_p=1,
        max_tokens=16000,
        timeout=None,
        max_retries=2
    ).with_structured_output(method="json_mode")

In [9]:
patient_text = """
Mujer de casi 32 años, natural y residente en la zona, en seguimiento en nuestra UCA de Vinaròs desde los 18 años (en 2005); en terapia conmigo desde 2014 (año de mi incorporación a la plaza).
De acuerdo a las notas de la historia clínica de papel y diferentes documentos consultados para la sesión clínica -a menudo desordenados, informes de distinta procedencia y cotejo con apuntes propios-, la paciente se inició en el consumo de tabaco y alcohol a los 12 años, en el cannabis a los 13 (diario desde los 15), en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol), y años más tarde consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox.
Lena dejó los estudios con 16 años, en 2º de la ESO, optando luego por trabajos muy precarios, erráticos, y sobre todo actividades marginales, entre ellas la prostitución hace 2 o 3 años, en Valencia.
Tiene reconocida una PNC (pensión no contributiva) por discapacidad del 73%, de la cual ella siempre se ha administrado el dinero, y hace un año fue inscrita por los Servicios Sociales de la localidad en un curso remunerado de administrativo, donde las condiciones de asistencia eran relativamente exigentes (horarios madrugadores, puntualidad, presencia, exposiciones), y a Lena le costaba bastante cumplir.
Mostraba esfuerzo, también motivada por el incentivo económico, pero había días que no acudía a clase, y ésta era también una de las razones para asistir con más frecuencia y compromiso a las citas médicas y psicológicas, que justificaban la ausencia de clase ese día.
Posteriormente ingresó voluntariamente en un centro de día para rehabilitación de tóxicos, con horarios poco compatibles con el curso de administrativo, sin perder la plaza gracias a una ILT (baja médica).
Respecto al entorno de Lena, la familia se caracteriza por su carencia de estructura.
Los padres de la paciente eran toxicómanos antes y durante su infancia, y ambos ya están fallecidos.
Ella fue acogida por la abuela materna, que también ha sido la persona que ha criado a una hermana 14 años menor, de un padre diferente (éste se encuentra vivo, también era toxicómano, fue presidiario, en la actualidad visita ocasionalmente a la familia, sobre todo a su hija, la hermana de Lena que ahora tiene 18 años).
Por tanto, desde el nacimiento la tutela de la paciente ha estado con los abuelos maternos, que actualmente cuentan con 68 años la mujer y 64 el marido (este hombre puede no ser el abuelo biológico de Lena, según una referencia de un informe de la historia clínica, y es la persona sobre la que actualmente ella deposita más hostilidad y rechazo).
La abuela es quien siempre se encarga de acompañar a Lena a los dispositivos, la ha rescatado en numerosas ocasiones de sitios hostiles y caóticos, ha supervisado muchas veces tratamientos y medicaciones, gestionado citas y visitas… es la principal figura de apego de la paciente, sin duda.
"""

In [10]:
prompt_val = tagging_prompt.invoke({"clinical_text": patient_text})
prompt_val

ChatPromptValue(messages=[HumanMessage(content='```\nYou are a highly specialized AI, an expert in Biomedical Natural Language Processing (NLP) with a focus on extracting information related to substance abuse from Spanish clinical text. Your primary task is Named Entity Recognition (NER), specifically identifying and categorizing mentions of Tobacco, Cannabis, Alcohol, and illicit/abused prescription Drugs within the provided text. You must also determine if the mention is positive or negated based on the sentence context. Your output should be a JSON formatted list of dictionaries.\nHere\'s the detailed breakdown of your process and the required output format:\n---\n## Task: Named Entity Recognition (NER) for Substance Abuse in Spanish Clinical Text with Negation Detection\n**Goal:** To accurately identify, categorize, and determine the sentiment (positive or negated) of named entities related to substance abuse within a given Spanish clinical text.\n**Entity Categories (Substance Ab

In [11]:
response = llm.invoke(prompt_val)
response

{'result': [{'entity': 'tabaco',
   'category': 'Tobacco',
   'start_index': 314,
   'end_index': 319,
   'sentiment': 'Positive'},
  {'entity': 'alcohol',
   'category': 'Alcohol',
   'start_index': 323,
   'end_index': 329,
   'sentiment': 'Positive'},
  {'entity': 'cannabis',
   'category': 'Cannabis',
   'start_index': 354,
   'end_index': 361,
   'sentiment': 'Positive'},
  {'entity': 'cocaína',
   'category': 'Drug',
   'start_index': 395,
   'end_index': 401,
   'sentiment': 'Positive'},
  {'entity': 'speed',
   'category': 'Drug',
   'start_index': 404,
   'end_index': 408,
   'sentiment': 'Positive'},
  {'entity': 'anfetaminas',
   'category': 'Drug',
   'start_index': 411,
   'end_index': 422,
   'sentiment': 'Positive'},
  {'entity': 'éxtasis',
   'category': 'Drug',
   'start_index': 425,
   'end_index': 431,
   'sentiment': 'Positive'},
  {'entity': 'heroína',
   'category': 'Drug',
   'start_index': 480,
   'end_index': 486,
   'sentiment': 'Positive'},
  {'entity': 'drog

In [12]:
import pandas as pd
from ast import literal_eval
dev_dataset = pd.read_csv(dev_dataset_path, sep='\t')
dev_dataset['trigger_annotations'] = dev_dataset['trigger_annotations'].apply(literal_eval)
dev_dataset['attr_annotations'] = dev_dataset['attr_annotations'].apply(literal_eval)
dev_dataset['response'] = ''
dev_dataset['entities'] = ''
dev_dataset.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
0,32073161_ES,"El 21 de enero de 2020, ingresó en el Hospital...","[{'label': 'Tobacco', 'off0': '569', 'off1': '...","[{'label': 'Duration', 'off0': '577', 'off1': ...",True,,
1,32277408_ES,﻿Un hombre de 63 años ingresó en el hospital a...,"[{'label': 'Tobacco', 'off0': '274', 'off1': '...",[],True,,
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False,,
3,32426200_ES,"Una mujer de 31 años, por lo demás sana, acudi...","[{'label': 'Alcohol', 'off0': '278', 'off1': '...","[{'label': 'Type', 'off0': '299', 'off1': '318...",True,,
4,32586958_ES,﻿Hombre de 21 años que acudió al servicio de u...,"[{'label': 'Tobacco', 'off0': '277', 'off1': '...",[],True,,


In [13]:
dev_dataset = dev_dataset[~dev_dataset['is_train']]

In [15]:
from tqdm.notebook import tqdm
tqdm.pandas()
count = 0
for index, row in tqdm(dev_dataset.iterrows(), total=dev_dataset.shape[0]):
    if row['response'] != '':
        continue
    responses = []
    texts = row['text'].split('\n\n')
    for text in texts:
        if text.strip() == '':
            continue
            
        prompt = tagging_prompt.invoke({"clinical_text": text.strip()})
        response = llm.invoke(prompt)
        responses.append(response)

    dev_dataset.at[index, 'response'] = responses
    count += 1

  0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
dev_dataset.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False,"[{'result': [{'entity': 'fumador', 'category':...",
5,casos_clinicos_cardiologia10,"Anamnesis\nSexo masculino, 79 años. Autoválido...","[{'label': 'Tobacco', 'off0': '126', 'off1': '...",[],False,"[{'response': []}, {'result': []}, {'result': ...",
16,casos_clinicos_cardiologia31,Varón de 61 años de edad que acude en junio de...,"[{'label': 'Tobacco', 'off0': '335', 'off1': '...",[],False,,
17,casos_clinicos_cardiologia4,Varón de 70 años.\n\nANTECEDENTES:\n- No reacc...,"[{'label': 'Tobacco', 'off0': '104', 'off1': '...",[],False,,
19,casos_clinicos_cardiologia64,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...","[{'label': 'Tobacco', 'off0': '123', 'off1': '...",[],False,,


In [17]:
dev_dataset.to_csv(predicted_results_path, index=False, sep='\t')

In [8]:
import pandas as pd
from ast import literal_eval
from tqdm.notebook import tqdm
tqdm.pandas()

dev_dataset = pd.read_csv('pred_dev_gpt41_langchain_gpt-4.1_few_shot.tsv', sep='\t')
#dev_dataset[''] = 
dev_dataset.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
0,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False,"[{'result': [{'entity': 'fumador', 'category':...","[{'entity': 'fumador', 'category': 'Tobacco', ..."
1,casos_clinicos_cardiologia10,"Anamnesis\nSexo masculino, 79 años. Autoválido...","[{'label': 'Tobacco', 'off0': '126', 'off1': '...",[],False,"[{'response': []}, {'result': []}, {'result': ...",[]
2,casos_clinicos_cardiologia31,Varón de 61 años de edad que acude en junio de...,"[{'label': 'Tobacco', 'off0': '335', 'off1': '...",[],False,NaN,NaN
3,casos_clinicos_cardiologia4,Varón de 70 años.\n\nANTECEDENTES:\n- No reacc...,"[{'label': 'Tobacco', 'off0': '104', 'off1': '...",[],False,NaN,NaN
4,casos_clinicos_cardiologia64,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...","[{'label': 'Tobacco', 'off0': '123', 'off1': '...",[],False,NaN,NaN


In [9]:
for index, row in tqdm(dev_dataset.iterrows(), total=dev_dataset.shape[0]):
    if row['response'] == '':
        continue

    entities = []
    for item in row['response']:
        if isinstance(item, dict):
            if 'entity' in item.keys():
                entities.append(item)
                continue
            for key in item.keys():
                if isinstance(item[key], dict):
                    entities.append(item[key])
                elif not isinstance(item[key], str) and not isinstance(item[key], int):
                    entities.extend(item[key])
                else:
                    print(index, item[key])
        else:
            print(row['response'])
    dev_dataset.at[index, 'entities'] = entities

  0%|          | 0/300 [00:00<?, ?it/s]

[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'result': [{'entity': 'fumador', 'category': 'Tobacco', 'start_index': 223, 'end_index': 230, 'sentiment': 'Positive'}]}]
[{'resul

TypeError: 'float' object is not iterable

In [10]:
dev_dataset.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
0,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False,"[{'result': [{'entity': 'fumador', 'category':...",[]
1,casos_clinicos_cardiologia10,"Anamnesis\nSexo masculino, 79 años. Autoválido...","[{'label': 'Tobacco', 'off0': '126', 'off1': '...",[],False,"[{'response': []}, {'result': []}, {'result': ...",[]
2,casos_clinicos_cardiologia31,Varón de 61 años de edad que acude en junio de...,"[{'label': 'Tobacco', 'off0': '335', 'off1': '...",[],False,NaN,NaN
3,casos_clinicos_cardiologia4,Varón de 70 años.\n\nANTECEDENTES:\n- No reacc...,"[{'label': 'Tobacco', 'off0': '104', 'off1': '...",[],False,NaN,NaN
4,casos_clinicos_cardiologia64,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...","[{'label': 'Tobacco', 'off0': '123', 'off1': '...",[],False,NaN,NaN


In [20]:
dev_dataset.to_csv(predicted_results_path, index=False, sep='\t')

In [21]:
dev_dataset.shape

(300, 7)

In [1]:
#Generate predictions in correct format

In [2]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [ ]:
import re

entities_list = []

for index, row in dev_dataset.iterrows():
    if row['response'] == '':
        break
        
    entities = row['entities']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        if isinstance(ent, dict) and 'entity' in ent.keys():
            term = ent['entity']
            label = ent['category']

            entity_list.extend(get_entities(row['filename'], text, term, label))        
        else:
            print(ent)
        
    entities_list.extend(entity_list)    

In [24]:
filenames = []
for index, row in dev_dataset.iterrows():
    if row['response'] == '':
        break
    filenames.append(row['filename'])

filenames = list(set(filenames))

In [25]:
len(filenames)

2

In [26]:
import pandas as pd

df_entities_list = pd.DataFrame.from_records(entities_list)
df_entities_list.drop_duplicates(inplace=True)
df_entities_list.head()

,filename,mark,label,off0,off1,span
0,32423911_ES,TOX,Tobacco,350,357,fumador


In [27]:
df_entities_list[['filename','label','off0','off1','span']].sort_values(by=['filename','off0','off1']).drop_duplicates(subset=['filename','off0','off1']).to_csv(predicted_entities_path, sep='\t', index=False, encoding ='utf-8')
df_entities_list.shape

(1, 6)

In [28]:
predicted_entities_path

'pred_dev_gpt41_langchain_gpt-4.1_few_shot_entities.tsv'